# SCOUT on KITTI

Walkthrough of the method on paired KITTI and Virtual KITTI frames. Each row is one driving frame. The real label is a missed-car score from a frozen detector on KITTI. The proxy is the same score on the Virtual KITTI clone of that frame. Images are not required: the repository already contains the paired table and the feature matrix.

One round of SCOUT:

1. Fit a failure model on the frames that already have a real label.
2. Keep a mutual-information shortlist of frames the model does not yet explain.
3. Re-rank that shortlist with a control variate that uses the Virtual KITTI score.
4. Spend the next real labels on that batch.


## 1. Get the code


In [ ]:
import os, subprocess, sys
from pathlib import Path

def find_root():
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        marker = cand / "data" / "kitti_vkitti" / "paired_detection_task_linked.csv"
        if marker.exists() and (cand / "scripts" / "kitti_vkitti" / "run_demo.py").exists():
            return cand
    return None

root = find_root()
if root is None and "google.colab" in sys.modules:
    subprocess.check_call(["git", "clone", "https://github.com/anjaliParashar/scout-code.git", "/content/scout-code"])
    root = Path("/content/scout-code")
if root is None:
    raise RuntimeError("Run this notebook from a checkout of scout-code, or open it in Colab.")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo:", root)


In [ ]:
import importlib.util, subprocess, sys
needed = ("torch", "sklearn", "pandas", "matplotlib", "joblib")
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "scipy", "scikit-learn", "matplotlib", "torch", "pandas", "joblib",
    ])
print("dependencies ready")


## 2. What one row of the pool is

`target_failure` is the real KITTI score. `proxy_failure` is the Virtual KITTI score for the same sequence and frame. The feature file `X_rich.npy` is aligned to this table. Higher failure means the detector missed more of the cars.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "axes.titlesize": 12})

task = pd.read_csv("data/kitti_vkitti/paired_detection_task_linked.csv")
features = np.load("data/kitti_vkitti/X_rich.npy")
y_t = task["target_failure"].to_numpy()
y_p = task["proxy_failure"].to_numpy()
print(f"paired frames: {len(task)}")
print(f"feature matrix: {features.shape}")
print(f"Pearson(proxy, target) = {np.corrcoef(y_p, y_t)[0, 1]:.3f}")
print(task[["sequence_id", "frame_id", "proxy_failure", "target_failure"]].head(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(5.2, 4.6), constrained_layout=True)
ax.scatter(y_p, y_t, s=12, alpha=0.45, c="#0072B2", edgecolors="none")
ax.set_xlabel("Virtual KITTI proxy failure")
ax.set_ylabel("KITTI target failure")
ax.set_title("Paired detector failures")
plt.show()


## 3. Run the method, the ablations, and the baselines

All five methods start from the same 8 real labels, then take 3 batches of 4. That budget is short so the cell finishes in a notebook. The full multi-seed sweeps are the scripts named at the end.

| Run | What changes |
| --- | --- |
| SCOUT | mutual-information shortlist, then a control variate that uses the proxy |
| β = 0 | same loop with the proxy term removed |
| MI only | shortlist by mutual information, no failure model |
| Random | uniform draws from the unlabeled frames |
| Importance sampling | draws with probability increasing in the proxy failure |


In [ ]:
from scripts.kitti_vkitti.run_demo import main

fig, summary = main([
    "--n-init", "8",
    "--n-acq-iters", "3",
    "--batch-size", "4",
    "--bnn-epochs", "40",
    "--mi-shortlist", "40",
    "--cv-n-jobs", "2",
    "--output-dir", "outputs/kitti_vkitti/notebook",
])
plt.show()
summary


## 4. How to read the plot

The dotted line is the end of the shared initial set. After that line, a curve that stays higher has spent its new real labels on frames with worse detector failures.

- The gap between SCOUT and β = 0 is the contribution of the Virtual KITTI control variate.
- MI only never fits a failure model, so it tests the exploration term on its own.
- Importance sampling trusts the proxy directly, with no mutual-information gate and no variance correction.
- Random ignores both the features and the proxy.

Longer runs, from the repository root:

```bash
python scripts/kitti_vkitti/ablate_local_cv.py
python scripts/kitti_vkitti/ablate_scenario_space.py
python scripts/kitti_vkitti/compare_beta_ablation.py --seeds 0 1 2
```
